# Teacher cache (remote GPU)
Thin wrapper over `scripts/teacher_cache.py`. Single GPU only.
Run `scripts/setup_remote.sh` first (or install the lock + bitsandbytes).
Needs internet ON (one-time teacher download from the Hub).

In [ ]:
REPO = '/content/ayanami-distill'  # adjust: clone path or uploaded zip root
PROMPTS = f'{REPO}/data/processed/pilot_50.jsonl'
# OUT includes the pinned revision (see configs/teachers/qwen-qwen3-8b.yaml):
OUT = 'data/teacher_cache/qwen-qwen3-8b/b968826d9c46dd6066d109eabc6255188de91218/'
# RELATIVE OUT: upload it back under the repo path. Shard range per session:
SHARDS = '0-0'  # e.g. '0-4' for five shards in one process, one model load
SHARD_SIZE = 100
SEED = 7

In [ ]:
# Hugging Face token via Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN).
# Never paste tokens in cells or files.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN set from Secrets')
except Exception as exc:
    print('no Secrets available (ok on Colab or other stores):', exc)
%env HF_HOME=/kaggle/tmp/hf_cache


In [ ]:
import subprocess, sys
# Preflight FIRST (no GPU/weights): abort here, not after the download.
pre = [sys.executable, f'{REPO}/scripts/teacher_cache.py', 'preflight',
       '--prompts', PROMPTS]
print(' '.join(pre))
r = subprocess.run(pre, cwd=REPO)
print('preflight returncode:', r.returncode)
assert r.returncode == 0
cmd = [sys.executable, f'{REPO}/scripts/teacher_cache.py', 'cache',
       '--prompts', PROMPTS, '--out', OUT,
       '--shards', SHARDS, '--shard-size', str(SHARD_SIZE),
       '--seed', str(SEED)]
print(' '.join(cmd))
r = subprocess.run(cmd, cwd=REPO)
print('returncode:', r.returncode)
assert r.returncode == 0

In [ ]:
# After the run: download OUT (shard_*.jsonl + manifest_*.json) and place it
# at <repo>/data/teacher_cache/qwen-qwen3-8b/, record hours in docs/GPU_LEDGER.md.
# Rerunning the same cell resumes (finished IDs are skipped).
!ls -la data/teacher_cache/qwen-qwen3-8b/ 2>/dev/null || echo 'run the cache cell first'